# Call Records Production Operations Notebook\n
\n
Use this notebook to run and verify the deployed ingestion pipeline end-to-end.\n
\n
Expected output contract from classifier:\n
INTENT: <Interested / Not Interested / Follow-up Needed / Inquiry / Already Enrolled / Spam / IGNORE>\n
SUMMARY: <Short English summary OR IGNORE>

In [ ]:
import os\n
import json\n
import requests

In [ ]:
BASE_URL = os.getenv("CALL_RECORDS_API_BASE", "http://localhost:8000")\n
INGEST_FOLDER = os.getenv("CALL_RECORDS_DRIVE_FOLDER", "")\n
INGEST_LIMIT = int(os.getenv("CALL_RECORDS_INGEST_LIMIT", "30"))\n
print("BASE_URL:", BASE_URL)

In [ ]:
health = requests.get(f"{BASE_URL}/health", timeout=30)\n
health.raise_for_status()\n
health.json()

In [ ]:
params = {"limit": INGEST_LIMIT}\n
if INGEST_FOLDER.strip():\n
    params["folder"] = INGEST_FOLDER.strip()\n
\n
response = requests.post(f"{BASE_URL}/api/ingest/run", params=params, timeout=300)\n
response.raise_for_status()\n
result = response.json()\n
print(json.dumps(result, indent=2))

In [ ]:
calls = requests.get(f"{BASE_URL}/api/dashboard/calls", params={"limit": 10}, timeout=30)\n
calls.raise_for_status()\n
rows = calls.json()\n
for row in rows:\n
    print("-", row.get("file_name"), "|", row.get("intent_category"), "|", row.get("summary"))